# Compare Outputs Between Base and Fine-Tuned Models
Uses Unsloth to load a fine-tuned version of Qwen 3.5 0.8B trained for binary query classification as EchoBot's guardian model.

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel

import os
os.environ["HF_HUB_OFFLINE"] = "1"

from dotenv import load_dotenv
load_dotenv()
SYSTEM_PROMPT = os.getenv("SYSTEM_PROMPT") 

max_seq_length = 2048

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


# Load Dataset

In [2]:
def formatting_func(sample):
    return {
        "prompt": (
            f"{SYSTEM_PROMPT}\n"
            f"User query: {sample['query']}\n"
            "Classification: "
        ),
        "completion": str(sample['label'])
    }

In [3]:
train_path = "../data/overfit/train.jsonl"
val_path = "../data/echobot/val.jsonl"
test_path = "../data/overfit/train.jsonl"

dataset = load_dataset(
    "json",
    data_files={
        "train": train_path,
        "validation": val_path,
        "test": test_path,
    },
)
print(dataset['train'].features)

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

{'id': Value('string'), 'query': Value('string'), 'label': Value('int64'), 'source': Value('string'), 'tag': Value('string')}


In [4]:
from collections import Counter

# format dataset with only 'prompt' and 'completion' collumns
formatted_train = dataset["train"].map(
    formatting_func,
    remove_columns=dataset["train"].column_names
    )
formatted_val = dataset["validation"].map(
    formatting_func,
    remove_columns=dataset["validation"].column_names
    )
formatted_test = dataset["test"].map(
    formatting_func,
    remove_columns=dataset["test"].column_names
    )

print(Counter(formatted_train["completion"]))
print(Counter(formatted_val["completion"]))
print(Counter(formatted_test["completion"]))


Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Counter({'0': 100, '1': 100})
Counter({'0': 4, '1': 4})
Counter({'0': 100, '1': 100})


# Compare Models

In [ ]:
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen3.5-0.8B",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    full_finetuning = False,
    dtype = torch.bfloat16,
    device_map= "auto",
)
FastLanguageModel.for_inference(base_model)

In [ ]:
# clean up before adding LoRA adapters
del base_model
del base_tokenizer

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [5]:
lora_model, lora_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "../adapters/eb_overfit-v1",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    full_finetuning = False,
    dtype = torch.bfloat16,
    device_map= "auto",
)
FastLanguageModel.for_inference(lora_model)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


==((====))==  Unsloth 2026.8.12: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100 80GB PCIe. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3_5ForConditionalGeneration(
      (model): Qwen3_5Model(
        (visual): Qwen3_5VisionModel(
          (patch_embed): Qwen3_5VisionPatchEmbed(
            (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
          )
          (pos_embed): Embedding(2304, 768)
          (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
          (blocks): ModuleList(
            (0-11): 12 x Qwen3_5VisionBlock(
              (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
              (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
              (attn): Qwen3_5VisionAttention(
                (qkv): Linear4bit(in_features=768, out_features=2304, bias=True)
                (proj): Linear4bit(in_features=768, out_features=768, bias=True)
              )
              (mlp): Qwen3_5VisionMLP(
                (linear_fc1): Linear4bit(in_features=768, out_features=3072, bias=True)
           

In [7]:
import re

def extract_label(generated_text):
    text = generated_text.strip()
    match = re.match(r"-?\d+", text)
    return match.group(0) if match else None

predictions = []
for example in formatted_test:
    inputs = lora_tokenizer(text=example["prompt"], return_tensors="pt").to(lora_model.device)
    with torch.no_grad():
        out = lora_model.generate(
            **inputs,
            max_new_tokens=8,
            do_sample=False,
            pad_token_id=lora_tokenizer.eos_token_id,
        )
    generated = lora_tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    predictions.append(extract_label(generated))

true_labels = formatted_test["completion"]
correct = sum(p == t for p, t in zip(predictions, true_labels))
print(f"Accuracy: {correct}/{len(true_labels)} = {correct/len(true_labels):.4f}")

Accuracy: 200/200 = 1.0000
